In [5]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import rasterio
from rasterio.transform import Affine
from skimage.transform import resize
from pathlib import Path
from tqdm import tqdm
import gc

# ---------------------------------------------------------
# 1. PARAMETERS & DIRECTORIES
# ---------------------------------------------------------
IMG_DIR = Path("../../data/tiled_data/images")
OUTPUT_DIR = Path("data/tiled_data/predictions/multiclass")
MODEL_PATH = "../../models/best_unet_model.keras"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256 
NUM_CLASSES = 25

# ---------------------------------------------------------
# 2. LOAD MODEL
# ---------------------------------------------------------
tf.keras.backend.clear_session()
MODEL = load_model(MODEL_PATH, compile=False)

image_files = list(IMG_DIR.glob("*.tif")) + list(IMG_DIR.glob("*.tiff"))
print(f"Found {len(image_files)} images. Resampling with CRS preservation...")

# ---------------------------------------------------------
# 3. LOOP THROUGH EACH IMAGE
# ---------------------------------------------------------
for img_path in tqdm(image_files, desc="Processing"):
    output_path = OUTPUT_DIR / f"{img_path.stem}pred.tif"
    
    with rasterio.open(img_path) as src:
        # Save original dimensions and CRS
        orig_h, orig_w = src.height, src.width
        orig_crs = src.crs
        orig_transform = src.transform
        
        # Read and prepare image data
        img_data = src.read()
        if img_data.shape[0] > 3:
            img_data = img_data[:3, ...]
        elif img_data.shape[0] == 1:
            img_data = np.repeat(img_data, 3, axis=0)
            
        # 1. Resize input for the model
        img_reshaped = np.moveaxis(img_data, 0, -1)
        img_resized = resize(img_reshaped, (TARGET_SIZE, TARGET_SIZE), preserve_range=True)
        
        # 2. Predict
        x_in = np.expand_dims(img_resized.astype(np.float32) / 255.0, 0)
        pred = MODEL.predict(x_in, verbose=0)[0]
        pred_class = (np.argmax(pred, axis=-1)+1).astype(np.uint8)

        # 3. CRITICAL: Scale the Transform to maintain CRS alignment
        # We calculate the scaling factor between original and target resolution
        rescale_factor_x = orig_w / TARGET_SIZE
        rescale_factor_y = orig_h / TARGET_SIZE
        
        # We multiply the original pixel size (A and E in Affine) by the rescale factor
        new_transform = orig_transform * Affine.scale(rescale_factor_x, rescale_factor_y)

        # 4. Create clean profile
        clean_profile = {
            'driver': 'GTiff',
            'dtype': 'uint8',
            'nodata': 0,
            'width': TARGET_SIZE,
            'height': TARGET_SIZE,
            'count': 1,
            'crs': orig_crs,
            'transform': new_transform,
            'compress': 'lzw',
            'tiled': False
        }

        # 5. Save the Prediction
        with rasterio.open(output_path, "w", **clean_profile) as dst:
            dst.write(pred_class, 1)

    tf.keras.backend.clear_session()
    gc.collect()

print(f"✅ Success! Predictions are 256x256 and aligned with CRS.")

Found 1026 images. Resampling with CRS preservation...


Processing: 100%|██████████| 1026/1026 [12:43<00:00,  1.34it/s]

✅ Success! Predictions are 256x256 and aligned with CRS.


In [ ]:
import matplotlib.pyplot as plt
import rasterio

# Pick one of your generated prediction files
sample_pred = "data/tiled_data/predictions/multiclass/some_image_pred.tif"

with rasterio.open(sample_pred) as src:
    mask = src.read(1)
    
    plt.figure(figsize=(8, 8))
    # 'tab20' or 'jet' are good for multiclass visibility
    plt.imshow(mask, cmap='tab20') 
    plt.colorbar(label='Class ID')
    plt.title("Multiclass Prediction Visualization")
    plt.show()

: 

Make the predictions in the folder to be binary (Waste or no waste)

Multiclass Model

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import rasterio
from rasterio.transform import Affine
from skimage.transform import resize
from pathlib import Path
from tqdm import tqdm
import gc

# ---------------------------------------------------------
# 1. PARAMETERS & DIRECTORIES
# ---------------------------------------------------------
IMG_DIR = Path("../../data/tiled_data/images")
OUTPUT_DIR = Path("data/tiled_data/predictions/multiclass")
MODEL_PATH = "../../models/best_unet_model.keras"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 256 
NUM_CLASSES = 10

# ---------------------------------------------------------
# 2. LOAD MODEL
# ---------------------------------------------------------
tf.keras.backend.clear_session()
MODEL = load_model(MODEL_PATH, compile=False)

image_files = list(IMG_DIR.glob("*.tif")) + list(IMG_DIR.glob("*.tiff"))
print(f"Found {len(image_files)} images. Resampling with CRS preservation...")

# ---------------------------------------------------------
# 3. LOOP THROUGH EACH IMAGE
# ---------------------------------------------------------
for img_path in tqdm(image_files, desc="Processing"):
    output_path = OUTPUT_DIR / f"{img_path.stem}pred.tif"
    
    with rasterio.open(img_path) as src:
        # Save original dimensions and CRS
        orig_h, orig_w = src.height, src.width
        orig_crs = src.crs
        orig_transform = src.transform
        
        # Read and prepare image data
        img_data = src.read()
        if img_data.shape[0] > 3:
            img_data = img_data[:3, ...]
        elif img_data.shape[0] == 1:
            img_data = np.repeat(img_data, 3, axis=0)
            
        # 1. Resize input for the model
        img_reshaped = np.moveaxis(img_data, 0, -1)
        img_resized = resize(img_reshaped, (TARGET_SIZE, TARGET_SIZE), preserve_range=True)
        
        # 2. Predict
        x_in = np.expand_dims(img_resized.astype(np.float32) / 255.0, 0)
        pred = MODEL.predict(x_in, verbose=0)[0]
        pred_class = (np.argmax(pred, axis=-1)+1).astype(np.uint8)

        # 3. CRITICAL: Scale the Transform to maintain CRS alignment
        # We calculate the scaling factor between original and target resolution
        rescale_factor_x = orig_w / TARGET_SIZE
        rescale_factor_y = orig_h / TARGET_SIZE
        
        # We multiply the original pixel size (A and E in Affine) by the rescale factor
        new_transform = orig_transform * Affine.scale(rescale_factor_x, rescale_factor_y)

        # 4. Create clean profile
        clean_profile = {
            'driver': 'GTiff',
            'dtype': 'uint8',
            'nodata': 0,
            'width': TARGET_SIZE,
            'height': TARGET_SIZE,
            'count': 1,
            'crs': orig_crs,
            'transform': new_transform,
            'compress': 'lzw',
            'tiled': False
        }

        # 5. Save the Prediction
        with rasterio.open(output_path, "w", **clean_profile) as dst:
            dst.write(pred_class, 1)

    tf.keras.backend.clear_session()
    gc.collect()

print(f"✅ Success! Predictions are 256x256 and aligned with CRS.")